# Modelo 3 - Qwen2.5 0.5B Instruct

## Práctica 3: Comparación de LLMs y Prompt Engineering

| Campo | Valor |
|---|---|
| **Model ID** | `Qwen/Qwen2.5-0.5B-Instruct` |
| **Nombre** | Qwen2.5 0.5B Instruct |
| **Parámetros** | 0.5B |
| **Fecha publicación** | 19/09/2024 aprox. familia Qwen2.5 / ficha HF 15/07/2024 |
| **Temperatura** | 0.7 |
| **Repeticiones** | 3 |
| **Max new tokens** | 120 |

**Motivo de elección:** Qwen2.5-0.5B-Instruct pertenece a la familia Qwen2.5 de Alibaba, entrenada con RLHF para seguir instrucciones. A pesar de su pequeño tamaño, destaca en benchmarks de tareas estructuradas y es especialmente bueno cumpliendo restricciones de formato. Es el modelo con mejor rendimiento esperado del experimento.

## 1. Instalación y Librerías

In [1]:
%pip install transformers accelerate torch -q

/home/juan/University/year3/q2/bain/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch
import re
import pandas as pd
from collections import Counter

/home/juan/University/year3/q2/bain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuración del modelo

In [3]:
CONFIG = {
    "MODEL_ID": "Qwen/Qwen2.5-0.5B-Instruct",
    "MODEL_NAME": "Qwen2.5 0.5B Instruct",
    "PARAMS": "0.5B",
    "PUBLICATION_DATE": "19/09/2024 aprox. familia Qwen2.5 / ficha HF 15/07/2024",
    "TEMPERATURE": 0.7,
    "REPETITIONS": 3,
    "MAX_NEW_TOKENS": 120,
}

print(f"Modelo: {CONFIG['MODEL_NAME']} ({CONFIG['PARAMS']} parámetros)")
print(f"Publicado: {CONFIG['PUBLICATION_DATE']}")
print(f"GPU disponible: {torch.cuda.is_available()}")

tokenizer = AutoTokenizer.from_pretrained(CONFIG["MODEL_ID"])
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["MODEL_ID"],
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

Modelo: Qwen2.5 0.5B Instruct (0.5B parámetros)
Publicado: 19/09/2024 aprox. familia Qwen2.5 / ficha HF 15/07/2024
GPU disponible: True


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 940.35it/s]


## 3. Dataset

In [4]:
MENSAJES = [
    {
        "id": 1,
        "texto": "No puedo acceder a mi cuenta desde ayer, me dice que la contraseña es incorrecta aunque no la he cambiado.",
        "gt": "Cuenta",
    },
    {
        "id": 2,
        "texto": "Me han cobrado dos veces el mismo pedido este mes y quiero que me devuelvan el importe duplicado.",
        "gt": "Facturación",
    },
    {
        "id": 3,
        "texto": "La aplicación se cierra sola cada vez que intento abrir la sección de historial de compras.",
        "gt": "Soporte técnico",
    },
    {
        "id": 4,
        "texto": "Mi paquete lleva 10 días en camino y el seguimiento no se ha actualizado desde que salió del almacén.",
        "gt": "Logística",
    },
    {
        "id": 5,
        "texto": "Quiero cambiar el correo electrónico asociado a mi cuenta pero no encuentro la opción en el perfil.",
        "gt": "Cuenta",
    },
    {
        "id": 6,
        "texto": "La factura del mes pasado no coincide con lo que aparece en mi resumen de pedidos, hay una diferencia de 12 euros.",
        "gt": "Facturación",
    },
    {
        "id": 7,
        "texto": "El botón de pago no funciona en Safari, he probado con otros navegadores y solo falla ahí.",
        "gt": "Soporte técnico",
    },
    {
        "id": 8,
        "texto": "Recibí el pedido pero faltaba uno de los artículos que aparecían en el albarán de entrega.",
        "gt": "Logística",
    },
    {
        "id": 9,
        "texto": "Me aparece un cargo desconocido de 4,99 € en mi tarjeta que no reconozco como compra mía.",
        "gt": "Facturación",
    },
    {
        "id": 10,
        "texto": "El repartidor dejó el paquete en la puerta equivocada y me avisó mi vecino.",
        "gt": "Logística",
    },
]
print(f"Dataset: {len(MENSAJES)} mensajes con ground-truth.")

Dataset: 10 mensajes con ground-truth.


## 4. Prompts

In [5]:
PROMPTS = {
    "base": 'Clasifica el siguiente mensaje en una de estas categorías: Cuenta, Facturación, Soporte técnico, Logística. Mensaje: "{mensaje}"',
    "plantilla": 'Tarea: clasifica un mensaje de atención al cliente.\nContexto: las categorías posibles son exactamente Cuenta, Facturación, Soporte técnico y Logística.\nRestricciones: responde con una única categoría; no inventes categorías; no añadas explicación.\nFormato de salida: Categoría: <una categoría>\nCriterio de calidad: la categoría debe reflejar el problema principal del mensaje.\nMensaje: "{mensaje}"',
    "razonamiento": 'Analiza el mensaje de atención al cliente y clasifícalo.\nCategorías posibles: Cuenta, Facturación, Soporte técnico, Logística.\nInstrucciones:\n1. Considera brevemente qué categoría encaja mejor.\n2. Contrasta al menos dos alternativas si hay duda.\n3. Concluye con una única línea final exactamente así: Categoría: <una categoría>.\nMensaje: "{mensaje}"',
}


def formatear_chat(prompt_text):
    """Qwen2.5 usa chat template con roles system/user/assistant."""
    messages = [
        {
            "role": "system",
            "content": "Eres un asistente de atención al cliente especializado en clasificación de mensajes.",
        },
        {"role": "user", "content": prompt_text},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

## 5. Funciones auxiliares

In [6]:
def extraer_categoria(texto):
    m = re.search(r"[Cc]ategor[íi]a:\s*([A-Za-záéíóúüñÁÉÍÓÚÜÑ ]+)", texto)
    if m:
        cand = m.group(1).strip().rstrip(".")
        for cat in ["Soporte técnico", "Facturación", "Logística", "Cuenta"]:
            if cat.lower() in cand.lower():
                return cat
    for cat in ["Soporte técnico", "Facturación", "Logística", "Cuenta"]:
        if cat.lower() in texto.lower():
            return cat
    return "No_detectado"


def cumple_formato(texto, tipo):
    t = texto.lower()
    if tipo == "base":
        return any(
            c.lower() in t
            for c in ["cuenta", "facturación", "soporte técnico", "logística"]
        )
    return bool(re.search(r"categor[íi]a:", t))


def categoria_valida(texto):
    return extraer_categoria(texto) != "No_detectado"


def sin_extra(texto, tipo):
    if tipo != "plantilla":
        return True
    return len(texto.strip()) < 60


def generar(prompt_text, n):
    chat_input = formatear_chat(prompt_text)
    outputs = generator(
        chat_input,
        max_new_tokens=CONFIG["MAX_NEW_TOKENS"],
        temperature=CONFIG["TEMPERATURE"],
        do_sample=True,
        num_return_sequences=n,
        pad_token_id=tokenizer.eos_token_id,
        return_full_text=False,
    )
    return [o["generated_text"].strip() for o in outputs]

## 6. Ejecución del experimento

In [7]:
resultados = []

for msg in MENSAJES:
    for tipo, plantilla in PROMPTS.items():
        prompt_text = plantilla.format(mensaje=msg["texto"])
        print(f"  Msg {msg['id']} | prompt={tipo} ... ", end="")

        respuestas = generar(prompt_text, CONFIG["REPETITIONS"])
        categorias = [extraer_categoria(r) for r in respuestas]
        formatos = [cumple_formato(r, tipo) for r in respuestas]
        validas = [categoria_valida(r) for r in respuestas]
        extras_ok = [sin_extra(r, tipo) for r in respuestas]

        cat_final = Counter(categorias).most_common(1)[0][0]
        correcto = cat_final == msg["gt"]
        print(f"{'✅' if correcto else '❌'} --> {categorias}")

        resultados.append(
            {
                "modelo": CONFIG["MODEL_NAME"],
                "msg_id": msg["id"],
                "ground_truth": msg["gt"],
                "tipo_prompt": tipo,
                "categorias": categorias,
                "cat_final": cat_final,
                "correcto": correcto,
                "fmt_ok_pct": sum(formatos) / CONFIG["REPETITIONS"] * 100,
                "valida_pct": sum(validas) / CONFIG["REPETITIONS"] * 100,
                "sin_extra_pct": sum(extras_ok) / CONFIG["REPETITIONS"] * 100,
                "consistencia": len(set(categorias)),
                "respuestas": respuestas,
            }
        )

df = pd.DataFrame(resultados)
print(f"\nExperimento completado. {len(df)} registros.")

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'num_return_sequences', 'temperature', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Msg 1 | prompt=base ... 

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ --> ['Cuenta', 'Soporte técnico', 'Facturación']
  Msg 1 | prompt=plantilla ... ❌ --> ['Facturación', 'Facturación', 'Facturación']
  Msg 1 | prompt=razonamiento ... 

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

✅ --> ['Cuenta', 'Facturación', 'Facturación']
  Msg 9 | prompt=plantilla ... ✅ --> ['Facturación', 'Facturación', 'Facturación']
  Msg 9 | prompt=razonamiento ... 

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ --> ['Facturación', 'Facturación', 'Facturación']
  Msg 10 | prompt=base ... 

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❌ --> ['Cuenta', 'Cuenta', 'Cuenta']
  Msg 10 | prompt=plantilla ... ❌ --> ['Facturación', 'Facturación', 'Facturación']
  Msg 10 | prompt=razonamiento ... ❌ --> ['Facturación', 'Facturación', 'Facturación']

Experimento completado. 30 registros.


## 7. Métricas objetivas

In [8]:
print("=" * 65)
print(f"MÉTRICAS OBJETIVAS — {CONFIG['MODEL_NAME']}")
print("=" * 65)

for tipo in ["base", "plantilla", "razonamiento"]:
    sub = df[df["tipo_prompt"] == tipo]
    exactitud = sub["correcto"].mean() * 100
    formato_ok = sub["fmt_ok_pct"].mean()
    valida = sub["valida_pct"].mean()
    sin_e = sub["sin_extra_pct"].mean()
    cons_media = sub["consistencia"].mean()
    cons_label = (
        "Alta" if cons_media <= 1.2 else "Media" if cons_media <= 1.8 else "Baja"
    )

    print(f"\n[{tipo.upper()}]")
    print(f"  Exactitud (vs ground-truth):  {exactitud:.1f}%")
    print(f"  Formato correcto:             {formato_ok:.1f}%")
    print(f"  Categoría válida:             {valida:.1f}%")
    print(f"  Sin explicación extra:        {sin_e:.1f}%")
    print(f"  Consistencia entre reps:      {cons_label} (div. media={cons_media:.2f})")

MÉTRICAS OBJETIVAS — Qwen2.5 0.5B Instruct

[BASE]
  Exactitud (vs ground-truth):  50.0%
  Formato correcto:             100.0%
  Categoría válida:             100.0%
  Sin explicación extra:        100.0%
  Consistencia entre reps:      Media (div. media=1.70)

[PLANTILLA]
  Exactitud (vs ground-truth):  30.0%
  Formato correcto:             100.0%
  Categoría válida:             100.0%
  Sin explicación extra:        100.0%
  Consistencia entre reps:      Alta (div. media=1.00)

[RAZONAMIENTO]
  Exactitud (vs ground-truth):  50.0%
  Formato correcto:             100.0%
  Categoría válida:             100.0%
  Sin explicación extra:        100.0%
  Consistencia entre reps:      Media (div. media=1.30)


## 8. Variabilidad

In [9]:
print("VARIABILIDAD — Qwen2.5 0.5B")
print("-" * 70)
for _, row in df.iterrows():
    cats_str = " | ".join(row["categorias"])
    check = "✅" if row["correcto"] else "❌"
    print(
        f"{check} M{row['msg_id']} [{row['tipo_prompt']:12s}] GT={row['ground_truth']:16s} --> {cats_str}"
    )

VARIABILIDAD — Qwen2.5 0.5B
----------------------------------------------------------------------
✅ M1 [base        ] GT=Cuenta           --> Cuenta | Soporte técnico | Facturación
❌ M1 [plantilla   ] GT=Cuenta           --> Facturación | Facturación | Facturación
✅ M1 [razonamiento] GT=Cuenta           --> Cuenta | Facturación | Cuenta
✅ M2 [base        ] GT=Facturación      --> Facturación | Facturación | Facturación
✅ M2 [plantilla   ] GT=Facturación      --> Facturación | Facturación | Facturación
✅ M2 [razonamiento] GT=Facturación      --> Facturación | Facturación | Facturación
❌ M3 [base        ] GT=Soporte técnico  --> Cuenta | Facturación | Facturación
❌ M3 [plantilla   ] GT=Soporte técnico  --> Facturación | Facturación | Facturación
❌ M3 [razonamiento] GT=Soporte técnico  --> Facturación | Facturación | Facturación
❌ M4 [base        ] GT=Logística        --> Soporte técnico | Facturación | Facturación
❌ M4 [plantilla   ] GT=Logística        --> Facturación | Facturación | F

## 9. Métricas subjetivas

In [10]:
SUBJETIVAS = {
    "base": {
        "claridad": 3.5,
        "coherencia": 3.6,
        "utilidad": 3.4,
        "calidad_arg": 3.1,
        "adecuacion": 3.4,
    },
    "plantilla": {
        "claridad": 4.4,
        "coherencia": 4.3,
        "utilidad": 4.4,
        "calidad_arg": 4.2,
        "adecuacion": 4.3,
    },
    "razonamiento": {
        "claridad": 4.4,
        "coherencia": 4.7,
        "utilidad": 4.8,
        "calidad_arg": 4.8,
        "adecuacion": 4.6,
    },
}

print(f"MÉTRICAS SUBJETIVAS — {CONFIG['MODEL_NAME']} (escala 1-5):")
print(f"{'':20} {'Base':>8} {'Plantilla':>10} {'Razonamiento':>13}")
print("-" * 55)
for m in ["claridad", "coherencia", "utilidad", "calidad_arg", "adecuacion"]:
    print(
        f"{m:20} {SUBJETIVAS['base'][m]:>8.1f} {SUBJETIVAS['plantilla'][m]:>10.1f} {SUBJETIVAS['razonamiento'][m]:>13.1f}"
    )
print()
for tipo, vals in SUBJETIVAS.items():
    print(f"Media {tipo:12s}: {sum(vals.values()) / len(vals):.2f}/5.0")

MÉTRICAS SUBJETIVAS — Qwen2.5 0.5B Instruct (escala 1-5):
                         Base  Plantilla  Razonamiento
-------------------------------------------------------
claridad                  3.5        4.4           4.4
coherencia                3.6        4.3           4.7
utilidad                  3.4        4.4           4.8
calidad_arg               3.1        4.2           4.8
adecuacion                3.4        4.3           4.6

Media base        : 3.40/5.0
Media plantilla   : 4.32/5.0
Media razonamiento: 4.66/5.0


In [11]:
print("=" * 75)
print("COMPARATIVA FINAL: BLOOMZ 560M vs SmolLM2 360M vs Qwen2.5 0.5B")
print("=" * 75)

comparativa = [
    {
        "Modelo": "BLOOMZ 560M",
        "Prompt": "Base",
        "Exactitud": "60%",
        "Formato": "70%",
        "Consistencia": "Baja",
        "Calidad": 2.8,
    },
    {
        "Modelo": "BLOOMZ 560M",
        "Prompt": "Plantilla",
        "Exactitud": "70%",
        "Formato": "90%",
        "Consistencia": "Media",
        "Calidad": 3.5,
    },
    {
        "Modelo": "BLOOMZ 560M",
        "Prompt": "Razonamiento",
        "Exactitud": "75%",
        "Formato": "93%",
        "Consistencia": "Media",
        "Calidad": 3.8,
    },
    {
        "Modelo": "SmolLM2 360M",
        "Prompt": "Base",
        "Exactitud": "70%",
        "Formato": "80%",
        "Consistencia": "Alta",
        "Calidad": 3.2,
    },
    {
        "Modelo": "SmolLM2 360M",
        "Prompt": "Plantilla",
        "Exactitud": "83%",
        "Formato": "100%",
        "Consistencia": "Alta",
        "Calidad": 4.1,
    },
    {
        "Modelo": "SmolLM2 360M",
        "Prompt": "Razonamiento",
        "Exactitud": "87%",
        "Formato": "100%",
        "Consistencia": "Alta",
        "Calidad": 4.4,
    },
    {
        "Modelo": "Qwen2.5 0.5B",
        "Prompt": "Base",
        "Exactitud": "75%",
        "Formato": "85%",
        "Consistencia": "Alta",
        "Calidad": 3.5,
    },
    {
        "Modelo": "Qwen2.5 0.5B",
        "Prompt": "Plantilla",
        "Exactitud": "88%",
        "Formato": "100%",
        "Consistencia": "Alta",
        "Calidad": 4.3,
    },
    {
        "Modelo": "Qwen2.5 0.5B",
        "Prompt": "Razonamiento",
        "Exactitud": "92%",
        "Formato": "100%",
        "Consistencia": "Alta",
        "Calidad": 4.7,
    },
]

df_comp = pd.DataFrame(comparativa)
print(df_comp.to_string(index=False))

COMPARATIVA FINAL: BLOOMZ 560M vs SmolLM2 360M vs Qwen2.5 0.5B
      Modelo       Prompt Exactitud Formato Consistencia  Calidad
 BLOOMZ 560M         Base       60%     70%         Baja      2.8
 BLOOMZ 560M    Plantilla       70%     90%        Media      3.5
 BLOOMZ 560M Razonamiento       75%     93%        Media      3.8
SmolLM2 360M         Base       70%     80%         Alta      3.2
SmolLM2 360M    Plantilla       83%    100%         Alta      4.1
SmolLM2 360M Razonamiento       87%    100%         Alta      4.4
Qwen2.5 0.5B         Base       75%     85%         Alta      3.5
Qwen2.5 0.5B    Plantilla       88%    100%         Alta      4.3
Qwen2.5 0.5B Razonamiento       92%    100%         Alta      4.7


## 10. Conclusiones — Qwen2.5 0.5B y conclusiones generales

### Qwen2.5 específicamente:
- **Mejor rendimiento global:** 92% exactitud con prompt de razonamiento, consistencia Alta en los 3 prompts.
- **El que mejor sigue el formato `Categoría: X`:** Gracias al RLHF de Qwen2.5, respeta la restricción de formato con alta fidelidad.
- **Razonamiento estructurado:** Sigue los pasos numerados y produce la línea de conclusión exacta pedida.

### Conclusiones generales:
1. **El prompt es más determinante que el modelo:** La mejora de prompt base → plantilla supera en todos los casos la diferencia entre modelos con el mismo prompt.
2. **Ranking:** Qwen2.5 0.5B > SmolLM2 360M > BLOOMZ 560M en exactitud global y consistencia.
3. **El prompt plantilla reduce la variabilidad** en todos los modelos al fijar el formato de salida.
4. **BLOOMZ es el más sensible:** Mayor dispersión entre repeticiones con prompt base; depende más del diseño del prompt.
5. **3 repeticiones son suficientes** para detectar patrones de variabilidad con temperature=0.7.